In [1]:
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
import scipy.optimize
import datetime
import sys

class gel_3D:
    def __init__(self, length=90.0, width=15.0, thickness=1.6, phi0=0.2, mu_bar=-0.05):
        self.phi0 = phi0
        print(f'Initial polymer volume fraction = {phi0:.2f} [dimensionless];', end=' ')
        self.mu_bar = mu_bar
        print(f'Normalized chemical potential = {mu_bar:.6f};', end=' ')

        T = 25 + 273.15
        K_B = 1.380649e-23
        V_m = 3e-29
        self.entropic_unit = K_B * T / V_m * 1e-6
        print(f'Entropic unit = {self.entropic_unit:.2f} [MPa];')

        self.gamma = 0.001
        self.chi = 0.4
        self.G = self.gamma * self.entropic_unit

        print(f'gamma = {self.gamma:.2e}; G = {self.G:.2f} MPa; chi = {self.chi:.3f}')

        vapor_pressure = 3.2e-3
        self.p0_bar = vapor_pressure / self.entropic_unit
        self.p_bar = self.p0_bar * np.exp(mu_bar)

        print(f'p_bar = {self.p_bar:.3e}')

        self.L = length
        self.d = thickness
        self.w = width

        self.filename_suffix = f'_phi0={self.phi0:.1f}_muBarAbs={np.abs(mu_bar):.6f}'

        # --- lambda isotrópico ---
        def auxIsotropic(s):
            return s*self.dH(s*s*s) + self.gamma

        self.lambda_iso = scipy.optimize.fsolve(auxIsotropic, phi0*1.1)[0]

        # --- lambda uniaxial target ---
        def auxUniaxial(s):
            return s*self.gamma + self.dH(s)

        self.lambda_target = scipy.optimize.fsolve(auxUniaxial, phi0*1.1)[0]

        print(f'lambda_iso = {self.lambda_iso:.3f}, lambda_target = {self.lambda_target:.3f}')

        def auxEnergyDensity(l1, l2, l3):
            J = l1*l2*l3
            phi = self.phi0/J
            return 0.5*self.G*(l1**2 + l2**2 + l3**2 - 3) + \
                   self.entropic_unit*((J-self.phi0)*np.log(1-phi) + self.phi0*self.chi*(1-phi)
                   - self.gamma*np.log(J) + (self.p_bar - self.mu_bar)*(J-self.phi0))

        self.reference_energy_density = auxEnergyDensity(
            self.lambda_iso, self.lambda_iso, self.lambda_iso
        )

    # -------------------------
    # FUNCIONES DEL MODELO
    # -------------------------

    def phi(self, J):
        return self.phi0 / J

    def H(self, J):
        return (J - self.phi0)*log(1-self.phi(J)) + \
               self.phi0*self.chi*(1-self.phi(J)) - \
               self.gamma*log(J) + \
               (self.p_bar - self.mu_bar)*(J-self.phi0)

    def dH(self, J):
        return self.phi(J) + np.log(1-self.phi(J)) + \
               self.chi*self.phi(J)**2 - self.gamma/J + \
               self.p_bar - self.mu_bar

    def Gfun(self, lamb):
        return (-self.dH(lamb)/lamb)*self.entropic_unit

    # -------------------------
    # ✅ MU_FUNC CORRECTO
    # -------------------------
    def mu_fun(self, lamb):
        """
        Calcula mu_bar consistente con equilibrio uniaxial
        """

        phi0 = self.phi0
        gamma = self.gamma
        chi = self.chi
        p0_bar = self.p0_bar

        def residual(mu_bar):
            p_bar = p0_bar * np.exp(mu_bar)
            phi = phi0 / lamb

            return (
                phi + np.log(1 - phi) + chi*phi**2
                - gamma/lamb
                + p_bar - mu_bar
            )

        mu_guess = -0.1
        mu_sol = scipy.optimize.fsolve(residual, mu_guess)[0]

        return mu_sol

    # -------------------------
    # ENERGÍA
    # -------------------------
    def W(self, F):
        J = Det(F)
        C_tensor = F.trans * F

        return 0.5*self.G*(Trace(C_tensor) - 3) + \
               self.entropic_unit*self.H(J) - \
               self.reference_energy_density

In [2]:
from ngsolve import x, y, z   # ✅ IMPORTANTE

class Solve_gel3d:
    def __init__(self, gel, order=1):
        self.gel = gel
        self.order = order
        self.start_time = datetime.datetime.now()  

    def add_mesh(self, mesh_file):        
        self.mesh = Mesh(mesh_file)
    
    def Space(self):
        self.fes = VectorH1(self.mesh, order=self.order, dirichlet="bonded|debonded")
        print('nDoF = {}'.format(self.fes.ndof))
        
    def model(self):
        u  = self.fes.TrialFunction()
        I = Id(self.mesh.dim)
        F = I + Grad(u)

        def negpart(var):
            return (sqrt(var**2)-var)*0.5        
        
        AA = 1e5

        # hydrogel model        
        self.a = BilinearForm(self.fes, symmetric=False)
        self.a += Variation(self.gel.W(F).Compile() * dx)

        # contacto (usa y directamente)
        self.a += Variation(AA*negpart(y+u[1])**2 * dx)
        
    def Solve_incremental_softening(self):
        self.Space()
        self.gfu = GridFunction(self.fes)

        # condición inicial uniaxial
        lambda_initial = 1.02

        if self.mesh.dim == 3:
            u0 = CoefficientFunction((0, (lambda_initial - 1.0)*y, 0))
        else:
            u0 = CoefficientFunction((0, (lambda_initial - 1.0)*y))

        self.gfu.Set(u0)

        # continuación en μ
        nIterations = 15

        lambda_list = np.linspace(lambda_initial, self.gel.lambda_target, nIterations)
        mu_list = [self.gel.mu_fun(la) for la in lambda_list]

        filename = 'gridfunctions/result' + \
            self.gel.filename_suffix + \
            "_order={}".format(self.order)

        tol = 1e-3
        maxits = 100

        self.model()

        for numIteration in range(nIterations):

            mu_i = mu_list[numIteration]

            print("*** Iteration #", numIteration, ". mu_bar = ", mu_i)

            if numIteration == nIterations - 1:
                tol = 1e-6
                maxits = 500

            self.gel.mu_bar = mu_i
            self.gel.p_bar = self.gel.p0_bar * np.exp(mu_i)

            self.gfu, _, _ = SolveNonlinearMinProblem(
                a=self.a,
                gfu=self.gfu,
                maxits=maxits,
                tol=tol,
                alpha=1e-2
            )

            self.gfu.Save(filename + '_iter=' + str(numIteration).zfill(2) + '.gfu')

            print("Total time elapsed =", datetime.datetime.now() - self.start_time)

In [3]:
def SolveNonlinearMinProblem(a, gfu, tol=1e-08, maxits=50, alpha=1.0):
    
    start_time = datetime.datetime.now()  

    res = gfu.vec.CreateVector()
    du  = gfu.vec.CreateVector()
    w   = gfu.vec.CreateVector()  # para line search
    
    precond = 'bddc'
    c = Preconditioner(a, precond)

    for it in range(maxits):

        with TaskManager():
            # Residuo actual
            a.Apply(gfu.vec, res)

            # Ensamblar Jacobiano
            a.AssembleLinearization(gfu.vec)

            c.Update()
            inv = CGSolver(a.mat, c.mat, maxsteps=1000)

            du.data = inv * res

        # ==============================
        # 🔹 LINE SEARCH (backtracking)
        # ==============================
        step = alpha
        success = False

        res_norm_old = sqrt(abs(InnerProduct(res, res)))

        for ls in range(10):  # máximo 10 intentos

            w.data = gfu.vec - step * du

            with TaskManager():
                a.Apply(w, res)
                res_norm_new = sqrt(abs(InnerProduct(res, res)))

            if res_norm_new < res_norm_old:
                success = True
                break

            step *= 0.5  # reducir paso

        if not success:
            print("⚠️ Line search falló, usando paso pequeño")
        
        # actualizar solución
        gfu.vec.data = w

        stopcritval = sqrt(abs(InnerProduct(du, res)))

        print("Newton iteration:", it, "Time elapsed =", datetime.datetime.now() - start_time)
        print("Residual norm =", res_norm_new, "Step =", step)

        if stopcritval < tol:
            break

    return gfu, stopcritval, it

In [4]:
# data = [dummy, L, w, d, phi0, abs(mu_bar)]   We will suppose that mu_bar is negative
data = ['', '90', '15.0', '1.62', '0.2', '0.0916'] 

order = 1

mesh_file = 'meshes/mesh0.vol.gz'

L = float(data[1])
w = float(data[2])   # ✅ CORREGIDO
d = float(data[3])   # ✅ CORREGIDO
phi0 = float(data[4])
mu_bar = - float(data[5])

print(f'L={L}, w={w}, d={d}, phi0={phi0}, mu_bar={mu_bar}')

gel = gel_3D(length=L, width=w, thickness=d, phi0=phi0, mu_bar=mu_bar)

modelling = Solve_gel3d(gel, order=order)
modelling.add_mesh(mesh_file)

L=90.0, w=15.0, d=1.62, phi0=0.2, mu_bar=-0.0916
Initial polymer volume fraction = 0.20 [dimensionless]; Normalized chemical potential = -0.091600; Entropic unit = 137.21 [MPa];
gamma = 1.00e-03; G = 0.14 MPa; chi = 0.400
p_bar = 2.128e-05
lambda_iso = 0.220, lambda_target = 0.405


/var/folders/6k/z6821vq94bz5_70q7ndgby400000gn/T/ipykernel_13491/1880549713.py:43: RuntimeWarning: The iteration is not making good progress, as measured by the 
 improvement from the last ten iterations.
  self.lambda_iso = scipy.optimize.fsolve(auxIsotropic, phi0*1.1)[0]


In [5]:
modelling = Solve_gel3d(gel, order=order)
modelling.add_mesh(mesh_file)
modelling.Solve_incremental_softening()

nDoF = 90978
*** Iteration # 0 . mu_bar =  -0.007753685111012042
Newton iteration: 0 Time elapsed = 0:00:01.867625
Residual norm = 240.84913386799036 Step = 0.0003125
Newton iteration: 1 Time elapsed = 0:00:03.791914
Residual norm = 240.64241693092097 Step = 0.00015625
Newton iteration: 2 Time elapsed = 0:00:05.490979
Residual norm = 240.43574411710648 Step = 0.0003125
Newton iteration: 3 Time elapsed = 0:00:07.207034
Residual norm = 240.34412222837557 Step = 0.00015625
Newton iteration: 4 Time elapsed = 0:00:08.828547
Residual norm = 240.3322394451663 Step = 0.000625
Newton iteration: 5 Time elapsed = 0:00:10.569389
Residual norm = 240.19673545078206 Step = 0.0003125
Newton iteration: 6 Time elapsed = 0:00:12.493999
Residual norm = 239.95144265124287 Step = 0.0003125
Newton iteration: 7 Time elapsed = 0:00:14.588448
Residual norm = 239.87365004689363 Step = 3.90625e-05
Newton iteration: 8 Time elapsed = 0:00:16.407751
Residual norm = 239.78880368769595 Step = 0.00015625
Newton iterati

KeyboardInterrupt: 